# Part A (supplement) — Calendar-Time Prepayment View

The KM survival curves in `partA_survival.ipynb` put all loans on the same
loan-age axis.  A macro event like the 2020 rate drop therefore appears at
different x-positions for different vintages: month 14 for a 2019 loan,
month 168 for a 2006 loan.

The two plots below use **calendar time** as the axis instead:

**Plot 1** — Aggregate monthly prepayment rate overlaid on the 30-year
mortgage rate (FRED). The rate-cycle humps that appear in the KM hazard show
up here as a single coherent spike at the date they actually happened.

**Plot 2** — Cross-sectional snapshot: for representative calendar years,
the monthly prepay rate by loan-age bucket. The *level* shift between years
is the rate-environment effect; the *shape* across age buckets is
seasoning/burnout. The two effects are visually separated here — the KM
hazard confounds them.

**Implementation note.** Both plots are computed entirely from the
`outcomes/` table (~70 MB) — no monthly panel scan required. We reconstruct
calendar-time prepayment counts from `first_obs_date` / `last_obs_date`
(which come from the monthly data at conversion time) and `event_type`.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

FIGURES_DIR = REPO_ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from src.credit_data import load_macro, load_outcomes

plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"
print("repo:", REPO_ROOT)


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
YEARS          = list(range(2006, 2023))   # in-sample vintages
SNAPSHOT_YEARS = [2009, 2012, 2017, 2020, 2022]
AGE_BREAKS     = [12, 36, 72, 120]
AGE_LABELS     = ["0-12", "12-36", "36-72", "72-120", "120+"]


In [ ]:
# Load the outcomes table — small (~70 MB), fits comfortably in RAM.
out = load_outcomes(
    years=YEARS,
    columns=["vintage_year", "first_obs_date", "last_obs_date", "event_type"],
).filter(pl.col("first_obs_date").is_not_null() & pl.col("last_obs_date").is_not_null())

print(f"outcomes loaded: {out.height:,} loans")
print(f"date range: {out['first_obs_date'].min()}  to  {out['last_obs_date'].max()}")


## Plot 1 — Aggregate monthly prepayment rate vs 30-year mortgage rate

For each calendar month `m`:
- `active[m]` = loans that have appeared by `m` and not yet left
  (cumulative starts minus cumulative prior-period ends).
- `prepaid[m]` = loans with `event_type == "prepaid"` whose `last_obs_date == m`.

Monthly hazard = `prepaid[m] / active[m]`.  This is exactly what the
monthly panel would give, reconstructed without loading it.


In [ ]:
# Monthly start/end/prepayment counts from the outcomes table.
starts   = out.group_by("first_obs_date").agg(pl.len().alias("n_starts"))
ends     = out.group_by("last_obs_date").agg(pl.len().alias("n_ends"))
prepaid  = (out.filter(pl.col("event_type") == "prepaid")
              .group_by("last_obs_date").agg(pl.len().alias("n_prepaid")))

# Build a complete calendar-month spine and join.
min_m = out["first_obs_date"].min()
max_m = out["last_obs_date"].max()
spine = pl.DataFrame({"month": pl.date_range(min_m, max_m, interval="1mo", eager=True)})

cal = (
    spine
    .join(starts.rename({"first_obs_date": "month"}),  on="month", how="left")
    .join(ends.rename({"last_obs_date":   "month"}),   on="month", how="left")
    .join(prepaid.rename({"last_obs_date": "month"}),  on="month", how="left")
    .fill_null(0)
    .with_columns([
        pl.col("n_starts").cum_sum().alias("cum_starts"),
        # ends lag by 1: a loan that ends in month m was active during m
        pl.col("n_ends").shift(1).fill_null(0).cum_sum().alias("cum_ended"),
    ])
    .with_columns(
        (pl.col("cum_starts") - pl.col("cum_ended")).alias("active")
    )
    .with_columns(
        (pl.col("n_prepaid") / pl.col("active") * 100).alias("cpr_pct")
    )
)

macro = load_macro().with_columns(pl.col("month").cast(pl.Date))
cal_pd = cal.join(macro, on="month", how="left").to_pandas()
cal_pd["month"] = pd.to_datetime(cal_pd["month"])

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(cal_pd["month"], cal_pd["cpr_pct"],
         color="tab:blue", lw=1.1, alpha=0.85, label="Monthly prepay rate")
ax1.set_ylabel("Monthly prepay rate (% of active loans)", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.set_ylim(bottom=0)

ax2 = ax1.twinx()
ax2.plot(cal_pd["month"], cal_pd["MORTGAGE30US"],
         color="tab:red", lw=1.6, ls="--", label="30y mortgage rate (FRED)")
ax2.set_ylabel("30-year mortgage rate (%)", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")
ax2.invert_yaxis()   # lower rate = higher prepay — invert so they move together

lines  = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
labels = ax1.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
ax1.legend(lines, labels, loc="upper left")
ax1.set_xlabel("Calendar month")
ax1.set_title(
    "Aggregate monthly prepayment rate vs 30-year mortgage rate\n"
    "(right axis inverted: rate drop and prepay spike both go up)",
    fontsize=11,
)
ax1.grid(alpha=0.25)
fig.savefig(FIGURES_DIR / "partA_calendar_prepay_vs_rate.png")
plt.show(); plt.close(fig)


## Plot 2 — Cross-sectional hazard by loan-age bucket at fixed calendar years

For each snapshot year `y`:
- **Active** loans = `first_obs_date.year ≤ y` and `last_obs_date.year ≥ y`.
- **Loan age at start of year** ≈ (`y` − `first_obs_date.year`) × 12 months.
- **Prepaid in year** = `event_type == "prepaid"` and `last_obs_date.year == y`.
- Monthly hazard ≈ prepaid-in-year / (active-loan-years × 12).

The shape across age buckets within each year reveals how seasoning and
burnout affect prepayment holding the rate environment roughly fixed.


In [ ]:
# Add convenience columns for year and approximate age at year-start.
out_snap = out.with_columns([
    pl.col("first_obs_date").dt.year().alias("orig_year"),
    pl.col("last_obs_date").dt.year().alias("end_year"),
])

rows = []
for snap_year in SNAPSHOT_YEARS:
    active = out_snap.filter(
        (pl.col("orig_year") <= snap_year) & (pl.col("end_year") >= snap_year)
    ).with_columns(
        # Loan age at start of snapshot year (months)
        ((snap_year - pl.col("orig_year")) * 12).cast(pl.Int32).alias("age_at_snap"),
        (
            (pl.col("event_type") == "prepaid") & (pl.col("end_year") == snap_year)
        ).cast(pl.Int8).alias("prepaid_this_year"),
    ).with_columns(
        pl.col("age_at_snap")
          .cut(AGE_BREAKS, labels=AGE_LABELS)
          .alias("age_bucket")
    )

    bucket_agg = (
        active.group_by("age_bucket")
        .agg([
            pl.len().alias("n_loans"),
            pl.col("prepaid_this_year").sum().alias("n_prepaid"),
        ])
        .with_columns([
            pl.lit(snap_year).alias("snap_year"),
            # Monthly hazard ≈ annual prepay fraction / 12
            (pl.col("n_prepaid") / pl.col("n_loans") / 12 * 100).alias("monthly_prepay_pct"),
        ])
    )
    rows.append(bucket_agg)

snap_pd = pl.concat(rows).to_pandas()

fig, ax = plt.subplots(figsize=(9, 5))
colors = plt.cm.plasma(np.linspace(0.1, 0.85, len(SNAPSHOT_YEARS)))

for color, yr in zip(colors, SNAPSHOT_YEARS):
    sub = snap_pd[snap_pd["snap_year"] == yr].copy()
    sub["age_bucket"] = pd.Categorical(sub["age_bucket"], categories=AGE_LABELS, ordered=True)
    sub = sub.sort_values("age_bucket")
    ax.plot(sub["age_bucket"], sub["monthly_prepay_pct"],
            marker="o", color=color, lw=1.6, label=str(yr))

ax.set_xlabel("Loan age bucket at start of snapshot year (months from origination)")
ax.set_ylabel("Approx monthly prepay rate (% within snapshot year)")
ax.set_title(
    "Cross-sectional prepayment hazard by loan age at fixed calendar years\n"
    "Level = rate environment   |   Shape = seasoning / burnout"
)
ax.legend(title="Calendar year")
ax.grid(alpha=0.3)
fig.savefig(FIGURES_DIR / "partA_cross_section_by_age.png")
plt.show(); plt.close(fig)
